In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/AI/'

In [ ]:
from tokenizers import BertWordPieceTokenizer

tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)

# 토크나이저 학습
tokenizer.train(
    # files='nsmc.txt',
    files=base_path + 'nsmc.txt',
    vocab_size=30000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
)

# 2. 임베딩

- 토큰화를 통해 얻은 ID 시퀀스 정보에 의미를 부여하는 과정
- `45` 정수 ID는 `44` 정수 ID나 `46` 정수 ID와 아무런 수학적 관계가 없음.
- 따라서, 각 ID 값들을 의미 있는 공간의 **좌표.** 즉, 벡터로 변환 할 필요가 있음.

## 2-1. 임베딩 종류와 발전 과정

- **어떻게 하면 단어의 의미를 벡터에 가장 잘 담을 수 있을까?**

### 2-1-0. 원핫 인코딩

- 가장 직관적이고 간단한 단어 표현 방식
- 단어 사전에 있는 단어의 수 만큼 벡터를 생성
- 표현 하고 싶은 단어의 인덱스 위치만 1로 표기, 나머지는 모두 0으로 처리
- 한계점
    1. **차원의 저주:** 단어 사전의 크기가 커지면 벡터도 그 만큼 커지지만, 대부분이 0인 낭비가 발생
    2. **의미 관계 표현의 부재**
        - 어떤 단어가 되었든, 단순히 단어 사전에 등장한 위치에만 의미가 있으므로 단어 간의 관계를 나타낼 수 가 없어짐.

In [ ]:
# 원핫 인코딩 예시
vocab = ['apple', 'banana', 'orange']

apple = [1, 0, 0]
banana = [0, 1, 0]
orange = [0, 0, 1]

### 2-1-1. 정적 임베딩

- 단어의 의미를 저차원의 `실수 벡터`에 압축하는 패러다임
    - **Word2Vec:** 단어의 의미는 주변 단어에 의해 결정된다는 아이디어 기반
        1. **CBOW (Continuous Bag-of-Words)**
            - 주변 단어들로 중심 단어를 예측하는 방식
            - “사과의 ㅇㅇㅇ 빨갛다” → ㅇㅇㅇ에 올 수 있는 단어는?
        2. **Skip-gram**
            - 중심 단어로 주변 단어들을 예측하는 방식
            - “ㅁㅁㅁ 창문을 OOO” → ㅁㅁㅁ와 ㅇㅇㅇ에 올 수 있는 단어는?
    - **GloVe (Global Vectors for Word Representation)**
        - 말뭉치 전체의 통계 정보를 사용해, 전체적인 통계를 먼저 계산하고 이를 벡터화
        

### 2-1-2. RNN/LSTMs

- 문맥적 임베딩을 시도한 첫 주류 모델
- 문장이 길어지면 **장기 의존성 문제**(앞쪽 정보 소실)와 **느린 속도** 문제 발생

### 2-1-3. Transformer / Positional Embedding / BERT

1. **Transformer** 
    - RNN의 순차 처리 방식에서 벗어남 → 문장 전체를 한 번에 병렬 처리하는 구조
    - **셀프 어텐션(Self-Attention):** 문장 내 모든 단어 간의 관계 중요도를 한 번에 계산
2. **Positional Embedding**
    - 단어의 순서 정보를 모델에 알려주기 위한 장치
    - 단어의 위치마다 고유한 벡터를 생성
    - 원래의 단어 의미 임베딩에 더해 줌
3. **BERT** (Bidirectional Encoder Representations from Transformers)
    - 트랜스포머의 인코더 구조만 사용
    - 문장의 **양방향** 문맥을 동시에 학습

## 2-2. 임베딩 층(Embedding Layer)이란?

- pytorch의 `nn.Embedding` 모듈을 활용하여 변환 과정을 진행 할 것
    - 학습 가능한 거대한 Lookup Table. (**가중치 행렬)**
- 행렬 구성: (단어 사전의 크기, 임베딩 벡터의 차원)
    - 행: 단어 사전에 있는 토큰 하나하나에 해당. (전체 샘플)
    - 열: 벡터의 차원을 나타냄. 차원의 크기가 클 수록 더 복잡한 관계를 벡터에 담을 수 있음.
    - ex) `45`번 ID의 토큰 `나는` 을 10차원 벡터화 한다면,
    이 가중치 행렬의 45번 행에 [1, 62, 4, 22, 81, 57, 7, 9, 10, 0] 형태의 벡터가 저장되는 셈

In [ ]:
import torch
import torch.nn as nn

vocab_size = 30000     # 챕터 1에서 정의한 단어 사전의 크기
embedding_dim = 768    # 각 단어를 표현할 벡터의 차원

# (30000, 768) 크기의 룩업 테이블(가중치 행렬)을 생성.
# 처음에는 임의의 값으로 초기화되어 있음.
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# 학습 과정에서 단어 간의 의미 관계를 포착하도록 업데이트 될 예정임.
# 현재는 초기화된 상태이므로, 임의의 값이 들어있음
print(embedding_layer.weight.shape)

# 예시: 단어 인덱스 45에 해당하는 단어의 임베딩 벡터 추출
word_index = 45
word_vector = embedding_layer.weight[word_index]
print(word_vector)

## 2-3. 미리보기: 임베딩 벡터, 어디에 쓸까?

- Word2Vec 등의 학습 과정을 통해 **단어의 의미와 문맥적 관계**를 벡터로 표현한다면,
- 이 벡터들을 활용해, **단어 간의 유사성**을 측정할 수 있음
    - 현재는 무작위 벡터값으로 초기화 되어 있음에 유의

### 2-3-1. 코사인 유사도

- 두 벡터가 고차원 공간에서 얼마나 `같은 방향`을 향하고 있는지를 측정하는 지표
1. 두 벡터 사이의 각도의 코사인 값을 계산
    - 각도가 작을 수록 코사인 값은 1에 가까워 짐. (같은 방향을 바라본다.)
    - 각도가 클 수록 코사인 값은 -1에 가까워 짐. (완전히 반대 방향을 바라본다.)
2. 수식은… 지금은 스킵

In [ ]:
import torch.nn.functional as F

# 챕터 1에서 만든 tokenizer와 방금 만든 embedding_layer를 사용
target_word = "영화"
top_k = 5

# 기준 단어의 ID와 벡터 조회
target_id = tokenizer.token_to_id(target_word)
target_vector = embedding_layer.weight[target_id]

# 전체 단어 벡터와 코사인 유사도 계산
all_vectors = embedding_layer.weight
similarities = F.cosine_similarity(target_vector.unsqueeze(0), all_vectors, dim=1)

# 유사도가 높은 Top-K 단어 찾기 (자기 자신 제외)
top_scores, top_indices = torch.topk(similarities, k=top_k + 1)

print(f"학습 전, '{target_word}'와 가장 유사한 단어 Top {top_k}:")
for i in range(1, top_k + 1):
    similar_word_id = top_indices[i].item()
    similar_word = tokenizer.id_to_token(similar_word_id)
    score = top_scores[i].item()
    print(f"{i}순위: {similar_word} (유사도: {score:.4f})")

# 현재 결과는 유의미한 단어가 나올 수 있을까?